In [ ]:
import polars as pl
from ewars_client import EWARSClient
import pandas as pd

In [ ]:
ewars_client = EWARSClient(user="drc_moh", password="@Dse2022", aid=5, client="tp-client")

In [ ]:
md = pl.read_csv("/home/leyregarrido/01_github_repos/openhexa-pipelines-drc-pnlp/dhis2_ewars_push/ewars_forms.csv")
ids = md.select(pl.col("id")).to_series().to_list()

In [ ]:
empty_ids = []
filled_ids = []
for id in ids:
    params = {
        "aid": ewars_client.aid,
        "username": ewars_client.user,
        "password": ewars_client.password,
        "from_date": "20260101",
        "to_date": "20260522",
        "form_id": id,
        "get_data_with_labels": False,
        "is_sub_form": False,
    }
    response = ewars_client._get(endpoint="reports", params=params)
    resp_value = response.json().get("value", None)
    if len(resp_value) == 0:
        empty_ids.append(id)
    else:
        filled_ids.append(id)

In [ ]:
still_empty_ids = []
for id in empty_ids:
    params = {
        "aid": ewars_client.aid,
        "username": ewars_client.user,
        "password": ewars_client.password,
        "from_date": "20250101",
        "to_date": "20251201",
        "form_id": id,
        "get_data_with_labels": False,
        "is_sub_form": False,
    }
    response = ewars_client._get(endpoint="reports", params=params)
    resp_value = response.json().get("value", None)
    if len(resp_value) == 0:
        still_empty_ids.append(id)
    else:
        filled_ids.append(id)

In [ ]:
def get_form_data(form_id: int) -> pd.DataFrame:
    params = {
        "aid": ewars_client.aid,
        "username": ewars_client.user,
        "password": ewars_client.password,
        "from_date": "2025-01-01",  # Only one date (same) to avoid time outs..
        "to_date": "2026-05-26",
        "form_id": form_id,
        "get_data_with_labels": False,
        "is_sub_form": False,
    }
    try:
        response = ewars_client._get(endpoint="reports", params=params)
        resp_value = response.json().get("value", None)
        if not resp_value:
            return pd.DataFrame()
        reports_df = pd.json_normalize(resp_value)
        reports_df.columns = ewars_client.clean_strings(input_list=reports_df.columns)
        reports_df.columns = reports_df.columns.str.replace("^data_", "", regex=True)  # remove extra "data_"
        return reports_df
    except Exception as e:
        print(f"Error while retrieving forms: {e}")
        return pd.DataFrame()

In [ ]:
all_info = []
for id in filled_ids:
    df = get_form_data(id)
    new_info = pl.DataFrame(
        {"id": id, "num_rows": len(df), "first_date": df["date"].min(), "last_date": df["date"].max()}
    )
    all_info.append(new_info)
    print(f"Form {id} has {len(df)} rows, from {df['date'].min()} to {df['date'].max()}")
forms_info = pl.concat(all_info)

In [ ]:
forms_info